## Fix your dataset here

In [1]:
import math

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import torch
from rkan.torch import PadeRKAN
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from datetime import datetime
from tqdm import tqdm
from torch.optim import LBFGS
import torch.nn as nn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


##### This part should be modified according to the dataset #####
# Assume taget label is in the column named 'target_label' and other columns are features
 
DATA = 'CB'  
data = pd.read_csv('CB.csv')
##################################################################

data.fillna(0, inplace=True)
y = data["target_label"].values
data.drop("target_label", axis=1, inplace=True)
X = data.values.astype(np.float32)

if y.dtype == "object":
    label_encoder = LabelEncoder()
    y = torch.tensor(label_encoder.fit_transform(y))
else:
    y = torch.tensor(y)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, stratify=y_train, test_size=0.1, random_state=42
)


scaler = StandardScaler()
X_train = torch.tensor(scaler.fit_transform(X_train)).to(device)
X_valid = torch.tensor(scaler.transform(X_valid)).to(device)
X_test = torch.tensor(scaler.transform(X_test)).to(device)


y_train = torch.nn.functional.one_hot(y_train.long(), num_classes=2).to(device).float()
y_valid = torch.nn.functional.one_hot(y_valid.long(), num_classes=2).to(device).float()
y_test = torch.nn.functional.one_hot(y_test.long(), num_classes=2).to(device).float()


input_shape = X_train.shape[1]
output_shape = y_train.shape[1]

print("Reduce number of features from %d to %d." % (data.shape[1], input_shape))

dataset = {
    "train_input": X_train,
    "train_label": y_train,
    "test_input": X_valid,
    "test_label": y_valid,
}


/home/defuser/miniconda3/envs/KAN/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reduce number of features from 556 to 556.


# Pade RKAN

In [2]:

EPOCHS = 1
TRIALS = 10

MAX_DEPTH = 2
MAX_NEURONS = 10

def fit(model, dataset, steps=100, loss_fn=None, lr=1., batch=-1):
    pbar = tqdm(range(steps), desc='description', ncols=100)
    optimizer = LBFGS(model.parameters(), lr=lr, history_size=10, line_search_fn="strong_wolfe", tolerance_grad=1e-32, tolerance_change=1e-32)

    results = {}
    results['train_loss'] = []
    results['test_loss'] = []

    if batch == -1 or batch > dataset['train_input'].shape[0]:
        batch_size = dataset['train_input'].shape[0]
        batch_size_test = dataset['test_input'].shape[0]
    else:
        batch_size = batch
        batch_size_test = batch

    global train_loss

    def closure():
        global train_loss
        optimizer.zero_grad()
        pred = model.forward(dataset['train_input'][train_id])
        train_loss = loss_fn(pred, dataset['train_label'][train_id])
        objective = train_loss
        objective.backward()
        return objective

    for _ in pbar:


        train_id = np.random.choice(dataset['train_input'].shape[0], batch_size, replace=False)
        test_id = np.random.choice(dataset['test_input'].shape[0], batch_size_test, replace=False)

        optimizer.step(closure)

        test_loss = loss_fn(model.forward(dataset['test_input'][test_id]), dataset['test_label'][test_id])

        results['train_loss'].append(torch.sqrt(train_loss).cpu().detach().numpy())
        results['test_loss'].append(torch.sqrt(test_loss).cpu().detach().numpy())
        pbar.set_description("| train_loss: %.2e | test_loss: %.2e " % (torch.sqrt(train_loss).cpu().detach().numpy(), torch.sqrt(test_loss).cpu().detach().numpy()))

    return results


class fKAN(nn.Module):
    def __init__(self, layers, orders1, orders2):
        super(fKAN, self).__init__()
        self.layers = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.layers.append(nn.Linear(layers[i], layers[i + 1]))
            if i < len(layers) - 2:
                self.layers.append(PadeRKAN(orders1[i], orders2[i]))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


def objective(trial):
    depth = trial.suggest_int("depth", 1, MAX_DEPTH)

    width = [trial.suggest_int(f"neurons_layer_{i}", 5, MAX_NEURONS, step=5) for i in range(depth)]
    orders1 = [trial.suggest_int(f"orders1_layer_{i}", 2, 6) for i in range(depth)]
    orders2 = [trial.suggest_int(f"orders2_layer_{i}", 2, 6) for i in range(depth)]
    width = [input_shape] + width + [output_shape]
    model = fKAN(width, orders1, orders2).to(device)

    history = fit(model, dataset, steps=EPOCHS, loss_fn=torch.nn.CrossEntropyLoss())

    y_score = model(X_valid).cpu()
    y_pred = (y_score > 0.5).int()

    return f1_score(y_valid.cpu(), y_pred, average="macro")


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=TRIALS)


best_params = study.best_params
print('best_params:', best_params)
depth = best_params["depth"]
width = [best_params[f"neurons_layer_{i}"] for i in range(depth)]
width = [input_shape] + width + [output_shape]
orders1 = [best_params[f"orders1_layer_{i}"] for i in range(depth)]
orders2 = [best_params[f"orders2_layer_{i}"] for i in range(depth)]

model = fKAN(width, orders1, orders2).to(device)
history = fit(model, dataset, steps=EPOCHS, loss_fn=torch.nn.CrossEntropyLoss())

y_score = model(X_test).cpu()
y_pred = (y_score > 0.5).int()


print(classification_report(y_test.cpu(), y_pred, target_names=label_encoder.classes_))
print("ROC-AUC: %.4f" % roc_auc_score(y_test.cpu(), y_pred))


results = study.trials_dataframe()
results.to_csv('results-rKAN2-%s-%s.csv' % (DATA, datetime.now()))

[I 2026-01-07 11:26:41,838] A new study created in memory with name: no-name-67904ade-046d-4ad6-835c-b210e0aa95a1
| train_loss: 1.67e-02 | test_loss: 1.96e+00 : 100%|██████████████████| 1/1 [00:00<00:00,  1.09it/s]
[I 2026-01-07 11:26:42,765] Trial 0 finished with value: 0.6777289084366254 and parameters: {'depth': 1, 'neurons_layer_0': 10, 'orders1_layer_0': 4, 'orders2_layer_0': 4}. Best is trial 0 with value: 0.6777289084366254.
| train_loss: 2.58e-01 | test_loss: 1.31e+00 : 100%|██████████████████| 1/1 [00:00<00:00,  3.60it/s]
[I 2026-01-07 11:26:43,051] Trial 1 finished with value: 0.6552250190694127 and parameters: {'depth': 1, 'neurons_layer_0': 10, 'orders1_layer_0': 6, 'orders2_layer_0': 3}. Best is trial 0 with value: 0.6777289084366254.
| train_loss: 6.27e-01 | test_loss: 9.22e-01 : 100%|██████████████████| 1/1 [00:00<00:00,  7.19it/s]
[I 2026-01-07 11:26:43,197] Trial 2 finished with value: 0.20046082949308755 and parameters: {'depth': 1, 'neurons_layer_0': 5, 'orders1_laye

best_params: {'depth': 1, 'neurons_layer_0': 10, 'orders1_layer_0': 6, 'orders2_layer_0': 6}


| train_loss: 4.96e-03 | test_loss: 1.89e+00 : 100%|██████████████████| 1/1 [00:00<00:00,  2.96it/s]

              precision    recall  f1-score   support

        band       0.78      0.62      0.69        63
      noband       0.67      0.84      0.74        62

   micro avg       0.71      0.73      0.72       125
   macro avg       0.72      0.73      0.72       125
weighted avg       0.72      0.73      0.72       125
 samples avg       0.70      0.73      0.71       125

ROC-AUC: 0.7169



/home/defuser/miniconda3/envs/KAN/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## FourierKAN

In [3]:
from fourierKAN import NaiveFourierKANLayer

MAX_DEPTH = 1
MAX_NEURONS = 5
MAX_GRID = 10


def fit(model, dataset, steps=100, loss_fn=None, lr=1., batch=-1):
    pbar = tqdm(range(steps), desc='description', ncols=100)
    optimizer = LBFGS(model.parameters(), lr=lr, history_size=10, line_search_fn="strong_wolfe", tolerance_grad=1e-32, tolerance_change=1e-32)

    results = {}
    results['train_loss'] = []
    results['test_loss'] = []

    if batch == -1 or batch > dataset['train_input'].shape[0]:
        batch_size = dataset['train_input'].shape[0]
        batch_size_test = dataset['test_input'].shape[0]
    else:
        batch_size = batch
        batch_size_test = batch

    global train_loss

    def closure():
        global train_loss
        optimizer.zero_grad()
        pred = model.forward(dataset['train_input'][train_id])
        train_loss = loss_fn(pred, dataset['train_label'][train_id])
        objective = train_loss
        objective.backward()
        return objective

    for _ in pbar:


        train_id = np.random.choice(dataset['train_input'].shape[0], batch_size, replace=False)
        test_id = np.random.choice(dataset['test_input'].shape[0], batch_size_test, replace=False)

        optimizer.step(closure)

        test_loss = loss_fn(model.forward(dataset['test_input'][test_id]), dataset['test_label'][test_id])

        results['train_loss'].append(torch.sqrt(train_loss).cpu().detach().numpy())
        results['test_loss'].append(torch.sqrt(test_loss).cpu().detach().numpy())
        pbar.set_description("| train_loss: %.2e | test_loss: %.2e " % (torch.sqrt(train_loss).cpu().detach().numpy(), torch.sqrt(test_loss).cpu().detach().numpy()))

    return results


class fKAN(nn.Module):
    def __init__(self, layers, gridsizes):
        super(fKAN, self).__init__()
        self.layers = nn.ModuleList()
        for i in range(len(layers) - 2):
            self.layers.append(NaiveFourierKANLayer(layers[i], layers[i + 1], gridsize=gridsizes[i]))
        self.layers.append(nn.Linear(layers[len(layers) - 2], layers[len(layers) - 1]))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


def objective(trial):
    depth = trial.suggest_int("depth", 1, MAX_DEPTH)
    width = [trial.suggest_int(f"neurons_layer_{i}", 5, MAX_NEURONS, step=5) for i in range(depth)]
    grids = [trial.suggest_int(f"grids_{i}", 1, MAX_GRID) for i in range(depth)]
    width = [input_shape] + width + [output_shape]
    model = fKAN(width, grids).to(device)

    history = fit(model, dataset, steps=EPOCHS, loss_fn=torch.nn.CrossEntropyLoss())

    y_score = model(X_valid).cpu()
    y_pred = (y_score > 0.5).int()

    return f1_score(y_valid.cpu(), y_pred, average="macro")


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=TRIALS)


best_params = study.best_params
print('best_params:', best_params)
depth = best_params["depth"]
width = [best_params[f"neurons_layer_{i}"] for i in range(depth)]
width = [input_shape] + width + [output_shape]
grids = [best_params[f"grids_{i}"] for i in range(depth)]

model = fKAN(width, grids).to(device)
history = fit(model, dataset, steps=EPOCHS, loss_fn=torch.nn.CrossEntropyLoss())

y_score = model(X_test).cpu()
y_pred = (y_score > 0.5).int()


print(classification_report(y_test.cpu(), y_pred, target_names=label_encoder.classes_))
print("ROC-AUC: %.4f" % roc_auc_score(y_test.cpu(), y_pred))


results = study.trials_dataframe()
results.to_csv('results-fourierKAN-%s-%s.csv' % (DATA, datetime.now()))


[I 2026-01-07 11:26:45,589] A new study created in memory with name: no-name-9eaf8899-b168-4a1a-bded-c80f328a6966
| train_loss: 5.50e-01 | test_loss: 7.43e-01 : 100%|██████████████████| 1/1 [00:00<00:00, 21.38it/s]
[I 2026-01-07 11:26:45,640] Trial 0 finished with value: 0.7419431845309143 and parameters: {'depth': 1, 'neurons_layer_0': 5, 'grids_0': 3}. Best is trial 0 with value: 0.7419431845309143.
| train_loss: 6.09e-01 | test_loss: 6.18e-01 : 100%|██████████████████| 1/1 [00:00<00:00, 25.11it/s]
[I 2026-01-07 11:26:45,683] Trial 1 finished with value: 0.0 and parameters: {'depth': 1, 'neurons_layer_0': 5, 'grids_0': 2}. Best is trial 0 with value: 0.7419431845309143.
| train_loss: 6.40e-01 | test_loss: 6.71e-01 : 100%|██████████████████| 1/1 [00:00<00:00, 17.42it/s]
[I 2026-01-07 11:26:45,745] Trial 2 finished with value: 0.8 and parameters: {'depth': 1, 'neurons_layer_0': 5, 'grids_0': 10}. Best is trial 2 with value: 0.8.
| train_loss: 6.43e-01 | test_loss: 6.63e-01 : 100%|█████

best_params: {'depth': 1, 'neurons_layer_0': 5, 'grids_0': 2}


| train_loss: 6.22e-01 | test_loss: 5.78e-01 : 100%|██████████████████| 1/1 [00:00<00:00, 26.31it/s]

              precision    recall  f1-score   support

        band       0.83      0.56      0.67        63
      noband       0.88      0.61      0.72        62

   micro avg       0.86      0.58      0.70       125
   macro avg       0.86      0.58      0.70       125
weighted avg       0.86      0.58      0.70       125
 samples avg       0.58      0.58      0.58       125

ROC-AUC: 0.7440



/home/defuser/miniconda3/envs/KAN/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


There are more example in 
https://github.com/aseslamian/TAbKAN/tree/main/runs